# Лабораторна робота №2. Частина 1
**Завдання:** 1. Завантажити дані VHI для всіх областей України (окрім середнього по країні) за допомогою `urllib`.
2. Додати дату та час до назви файлу. Запобігти повторному завантаженню.

In [3]:
import urllib.request
import os
from datetime import datetime

# Створення папки для даних
folder_path = "data"
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

def download_noaa_data():
    current_year = datetime.now().year
    
    # Перебираємо індекси від 1 до 27 (0 пропускаємо, як вимагає ТЗ)
    for province_id in range(1, 28):
        # Перевірка на наявність вже завантаженого файлу для цієї області
        existing_files = [f for f in os.listdir(folder_path) if f.startswith(f"vhi_id_{province_id}_")]
        if existing_files:
            print(f"Файл для області {province_id} вже існує: {existing_files[0]}. Пропускаємо.")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2={current_year}&type=Mean"
        
        # Формування імені файлу з датою та часом
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = f"vhi_id_{province_id}_{timestamp}.csv"
        filepath = os.path.join(folder_path, filename)
        
        print(f"Завантажую дані для області {province_id}...")
        try:
            urllib.request.urlretrieve(url, filepath)
        except Exception as e:
            print(f"Помилка завантаження для області {province_id}: {e}")

download_noaa_data()

Файл для області 1 вже існує: vhi_id_1_2026-05-19_14-32-56.csv. Пропускаємо.
Файл для області 2 вже існує: vhi_id_2_2026-05-19_14-33-02.csv. Пропускаємо.
Файл для області 3 вже існує: vhi_id_3_2026-05-19_14-33-03.csv. Пропускаємо.
Файл для області 4 вже існує: vhi_id_4_2026-05-19_14-33-05.csv. Пропускаємо.
Файл для області 5 вже існує: vhi_id_5_2026-05-19_14-33-06.csv. Пропускаємо.
Файл для області 6 вже існує: vhi_id_6_2026-05-19_14-33-07.csv. Пропускаємо.
Файл для області 7 вже існує: vhi_id_7_2026-05-19_14-33-08.csv. Пропускаємо.
Файл для області 8 вже існує: vhi_id_8_2026-05-19_14-33-09.csv. Пропускаємо.
Файл для області 9 вже існує: vhi_id_9_2026-05-19_14-33-11.csv. Пропускаємо.
Файл для області 10 вже існує: vhi_id_10_2026-05-19_14-33-12.csv. Пропускаємо.
Файл для області 11 вже існує: vhi_id_11_2026-05-19_14-33-13.csv. Пропускаємо.
Файл для області 12 вже існує: vhi_id_12_2026-05-19_14-33-14.csv. Пропускаємо.
Файл для області 13 вже існує: vhi_id_13_2026-05-19_14-33-15.csv. Проп

**Завдання:** 1. Зчитати завантажені текстові файли у pandas dataframe.
2. Здійснити data cleaning (прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст).
3. Змінити індекси областей за українською абеткою (1 - Вінницька).

In [5]:
import pandas as pd

# Словник для заміни індексів NOAA на українську абетку
province_mapping = {
    1: (22, 'Черкаська'), 2: (24, 'Чернігівська'), 3: (23, 'Чернівецька'),
    4: (25, 'АР Крим'), 5: (3, 'Дніпропетровська'), 6: (4, 'Донецька'),
    7: (8, 'Івано-Франківська'), 8: (19, 'Харківська'), 9: (20, 'Херсонська'),
    10: (21, 'Хмельницька'), 11: (9, 'Київська'), 12: (26, 'м. Київ'),
    13: (10, 'Кіровоградська'), 14: (11, 'Луганська'), 15: (12, 'Львівська'),
    16: (13, 'Миколаївська'), 17: (14, 'Одеська'), 18: (15, 'Полтавська'),
    19: (16, 'Рівненська'), 20: (27, 'м. Севастополь'), 21: (17, 'Сумська'),
    22: (18, 'Тернопільська'), 23: (6, 'Закарпатська'), 24: (1, 'Вінницька'),
    25: (2, 'Волинська'), 26: (7, 'Запорізька'), 27: (5, 'Житомирська')
}

def load_and_clean_data(folder="data"):
    frames = []
    
    for filename in os.listdir(folder):
        if not filename.endswith(".csv"): continue
            
        filepath = os.path.join(folder, filename)
        old_id = int(filename.split('_')[2])
        
        # index_col=False запобігає зсуву колонок через кому в кінці рядка
        df = pd.read_csv(filepath, header=1, index_col=False)
        
        # Очистка назв колонок
        df.columns = [col.replace('<br>', '').strip() for col in df.columns]
        expected_cols = ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
        df = df[[c for c in expected_cols if c in df.columns]]
        
        # Очистка даних від HTML-тегів та некоректних типів
        if not df.empty and 'year' in df.columns:
            df['year'] = df['year'].astype(str).replace({'<tt><pre>': '', '</pre></tt>': ''}, regex=True).str.strip()
            df['year'] = pd.to_numeric(df['year'], errors='coerce')
            df['week'] = pd.to_numeric(df['week'], errors='coerce')
            df = df.dropna(subset=['year', 'week'])
            df['year'] = df['year'].astype(int)
            df['week'] = df['week'].astype(int)
            
        # Заміна відсутніх значень NOAA (-1) та видалення цих рядків
        df = df.replace(-1, pd.NA).dropna()
        
        # Додавання нових індексів
        new_id, prov_name = province_mapping.get(old_id, (old_id, "Невідомо"))
        df['area_id'] = new_id
        df['area_name'] = prov_name
        
        frames.append(df)
        
    final_df = pd.concat(frames, ignore_index=True)
    final_df = final_df.sort_values(by=['area_id', 'year', 'week']).reset_index(drop=True)
    return final_df[['area_id', 'area_name', 'year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]

# Створення глобального датафрейму
df = load_and_clean_data()
print("Дані успішно завантажено та очищено. Розмір:", df.shape)
display(df.head())

Дані успішно завантажено та очищено. Розмір: (60939, 9)


,area_id,area_name,year,week,SMN,SMT,VCI,TCI,VHI
0,1,Вінницька,1982,1,0.068,263.59,63.47,28.34,45.9
1,1,Вінницька,1982,2,0.074,265.78,67.62,23.05,45.34
2,1,Вінницька,1982,3,0.076,267.19,69.37,20.4,44.88
3,1,Вінницька,1982,4,0.075,268.57,65.26,17.93,41.6
4,1,Вінницька,1982,5,0.072,269.24,58.58,20.0,39.29


### Процедури формування вибірок
**Завдання:** Ряд VHI для області за вказаний рік.

In [11]:
def get_vhi_by_year(dataframe, province_id, year):
    result = dataframe[(dataframe['area_id'] == province_id) & (dataframe['year'] == year)]
    return result[['week', 'VHI']]

print("Вибірка: Ряд VHI для Вінницької області (ID: 1) за вказаний рік:")
display(get_vhi_by_year(df, 1, 1982).head())

Вибірка: Ряд VHI для Вінницької області (ID: 1) за вказаний рік:


,week,VHI
0,1,45.9
1,2,45.34
2,3,44.88
3,4,41.6
4,5,39.29


**Завдання:** Ряд VHI за вказаний діапазон років для вказаних областей.

In [13]:
def get_vhi_by_years_range_and_areas(dataframe, province_ids, year_start, year_end):
    result = dataframe[
        (dataframe['area_id'].isin(province_ids)) & 
        (dataframe['year'] >= year_start) & 
        (dataframe['year'] <= year_end)
    ]
    return result[['area_id', 'area_name', 'year', 'week', 'VHI']]

print("Вибірка: Ряд VHI для обраних областей за вказаний діапазон:")
display(get_vhi_by_years_range_and_areas(df, [9, 14], 2018, 2019).head(10))

Вибірка: Ряд VHI для обраних областей за вказаний діапазон:


,area_id,area_name,year,week,VHI
19878,9,Київська,2018,1,40.82
19879,9,Київська,2018,2,43.65
19880,9,Київська,2018,3,48.62
19881,9,Київська,2018,4,51.77
19882,9,Київська,2018,5,52.08
19883,9,Київська,2018,6,51.61
19884,9,Київська,2018,7,49.43
19885,9,Київська,2018,8,47.62
19886,9,Київська,2018,9,47.09
19887,9,Київська,2018,10,46.52


**Завдання:** Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [18]:
def get_vhi_statistics(dataframe, province_ids, years):
    result = dataframe[
        (dataframe['area_id'].isin(province_ids)) & 
        (dataframe['year'].isin(years))
    ]
    
    stats = {
        'Мінімум VHI': result['VHI'].min(),
        'Максимум VHI': result['VHI'].max(),
        'Середнє VHI': round(result['VHI'].mean(), 2),
        'Медіана VHI': result['VHI'].median()
    }
    
    return pd.DataFrame([stats])

print("Вибірка: Статистика VHI для вказаних областей за вказані роки:")
display(get_vhi_statistics(df, [12, 19], [2015, 2020]))

Вибірка: Статистика VHI для вказаних областей за вказані роки:


,Мінімум VHI,Максимум VHI,Середнє VHI,Медіана VHI
0,20.89,66.26,46.51,46.2
